<a href="https://colab.research.google.com/github/patkbung/-Assignment-Voting-/blob/main/DOG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("dilakshanchandrasena/dog-breed-classification")

print(path)
print(os.listdir(path))

100%|██████████| 689M/689M [00:17<00:00, 40.5MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/dilakshanchandrasena/dog-breed-classification/versions/1
['DATASET']


In [ ]:
DATA_DIR = path + "/DATASET"
print(os.listdir(DATA_DIR))

['train_images', 'test_images', 'test_data.csv', 'train_data.csv']


In [ ]:
TRAIN_DIR = DATA_DIR + "/train_images"
print(os.listdir(TRAIN_DIR)[:10])  # ดูแค่ 10 อันแรก

['007ff9a78eba2aebb558afea3a51c469.jpg', '66ff9ad63b61820d7472752accee8919.jpg', '3b1b257e380f47c09ffc4f4aa7011a2e.jpg', 'e79011daac807552f798aa1effb60ee4.jpg', '0ef07f4a6706a04af9ff354e263a28b3.jpg', 'bda8225dd3edf5e413f18e52fdcf5050.jpg', '30387c787edea41ec448ac6bda7822cd.jpg', 'b8e77ec7272a78a3340dab5513917a85.jpg', '266115adee245e31c3b8ee4860db376f.jpg', 'e316925eb1cf7cdeb1ffaab7424e231d.jpg']


In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR + "/train_data.csv")
print(df.head(10))
print(f"\nจำนวนรูปทั้งหมด: {len(df)}")
print(f"จำนวนสายพันธุ์: {df['breed'].nunique()}")

                                 id               breed
0  000bec180eb18c7604dcecc8fe0dba07         boston_bull
1  001513dfcb2ffafc82cccf4d8bbaba97               dingo
2  001cdf01b096e06d78e9e5112d419397            pekinese
3  00214f311d5d2247d5dfe4fe24b2303d            bluetick
4  0021f9ceb3235effd7fcde7f7538ed62    golden_retriever
5  002211c81b498ef88e1b40b9abf84e1d  bedlington_terrier
6  00290d3e1fdd27226ba27a8ce248ce85  bedlington_terrier
7  002a283a315af96eaea0e28e7163b21b              borzoi
8  003df8b8a8b05244b1d920bb6cf451f9             basenji
9  0042188c895a2f14ef64a918ed9c7b64  scottish_deerhound

จำนวนรูปทั้งหมด: 10222
จำนวนสายพันธุ์: 120


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# เพิ่ม path ของรูปเข้าไปใน dataframe
df['filename'] = TRAIN_DIR + "/" + df['id'] + ".jpg"

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = datagen.flow_from_dataframe(
    df,
    x_col='filename',
    y_col='breed',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_dataframe(
    df,
    x_col='filename',
    y_col='breed',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

NUM_CLASSES = len(train_data.class_indices)
print(f"✅ พร้อมแล้ว! จำนวน class: {NUM_CLASSES}")

Found 8178 validated image filenames belonging to 120 classes.
Found 2044 validated image filenames belonging to 120 classes.
✅ พร้อมแล้ว! จำนวน class: 120


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# สร้าง Model
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model พร้อมแล้ว!")

# Train!
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ Model พร้อมแล้ว!
Epoch 1/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 194s 678ms/step - accuracy: 0.2864 - loss: 3.0379 - val_accuracy: 0.6018 - val_loss: 1.5106
Epoch 2/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 546ms/step - accuracy: 0.5253 - loss: 1.7127 - val_accuracy: 0.6375 - val_loss: 1.2425
Epoch 3/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 548ms/step - accuracy: 0.5866 - loss: 1.4594 - val_accuracy: 0.6365 - val_loss: 1.2060
Epoch 4/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 138s 541ms/step - accuracy: 0.6192 - loss: 1.3172 - val_accuracy: 0.6624 - val_loss: 1.1466
Epoch 5/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 548ms/step - accuracy: 0.6363 - loss: 1.2320 - val_accuracy: 0.6585 - val_loss: 1.1343
Epoch 6/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 138s 540ms/step - accuracy: 0.6568 - loss: 1.1491 - val_accuracy: 0.6585 - val_loss: 1.1361
Epoch 7/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 142s 556ms/step - accuracy: 0.6740 - loss: 1.0930 - val_accuracy: 0.6742 - val_loss: 1.1146
Epoch 8/

In [ ]:
# ดูผลลัพธ์
print(f"Train Accuracy: {max(history.history['accuracy']):.2%}")
print(f"Val Accuracy: {max(history.history['val_accuracy']):.2%}")

Train Accuracy: 74.00%
Val Accuracy: 69.96%


In [ ]:
import tensorflow as tf

In [ ]:
# Unfreeze layer ล่างๆ ของ MobileNetV2
base.trainable = True

# แต่ freeze ส่วนบนไว้ก่อน เปิดแค่ layer ท้ายๆ
for layer in base.layers[:-30]:
    layer.trainable = False

# ใช้ learning rate เล็กมากๆ ตอน fine-tune
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train ต่ออีกรอบ
history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

Epoch 1/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 177s 619ms/step - accuracy: 0.6017 - loss: 1.3998 - val_accuracy: 0.6898 - val_loss: 1.0466
Epoch 2/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 141s 549ms/step - accuracy: 0.6641 - loss: 1.1335 - val_accuracy: 0.6898 - val_loss: 1.1108
Epoch 3/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 548ms/step - accuracy: 0.6850 - loss: 1.0590 - val_accuracy: 0.6898 - val_loss: 1.0722
Epoch 4/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 141s 552ms/step - accuracy: 0.7045 - loss: 0.9948 - val_accuracy: 0.6908 - val_loss: 1.0624


In [ ]:
print(f"Train Accuracy: {max(history2.history['accuracy']):.2%}")
print(f"Val Accuracy: {max(history2.history['val_accuracy']):.2%}")

Train Accuracy: 70.45%
Val Accuracy: 69.08%


**Reasoning**:
First, I will analyze the accuracy and loss of the MobileNetV2 model before and after fine-tuning, using the `history` and `history2` objects from the previous training cells.



In [ ]:
print("### MobileNetV2 Performance (Initial Training)")
print(f"Train Accuracy: {max(history.history['accuracy']):.2%}")
print(f"Val Accuracy: {max(history.history['val_accuracy']):.2%}")
print(f"Train Loss: {min(history.history['loss']):.4f}")
print(f"Val Loss: {min(history.history['val_loss']):.4f}")

print("\n### MobileNetV2 Performance (After Fine-tuning)")
print(f"Train Accuracy: {max(history2.history['accuracy']):.2%}")
print(f"Val Accuracy: {max(history2.history['val_accuracy']):.2%}")
print(f"Train Loss: {min(history2.history['loss']):.4f}")
print(f"Val Loss: {min(history2.history['val_loss']):.4f}")

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# เปลี่ยน base model
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

model2 = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history3 = model2.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(patience=2, factor=0.5)  # ปรับ learning rate อัตโนมัติ
    ]
)

Epoch 1/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 182s 625ms/step - accuracy: 0.0093 - loss: 5.3444 - val_accuracy: 0.0088 - val_loss: 4.7875 - learning_rate: 0.0010
Epoch 2/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 137s 534ms/step - accuracy: 0.0076 - loss: 4.9167 - val_accuracy: 0.0093 - val_loss: 4.7867 - learning_rate: 0.0010
Epoch 3/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 151s 589ms/step - accuracy: 0.0108 - loss: 4.8040 - val_accuracy: 0.0093 - val_loss: 4.7858 - learning_rate: 0.0010
Epoch 4/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 136s 532ms/step - accuracy: 0.0122 - loss: 4.7885 - val_accuracy: 0.0113 - val_loss: 4.7852 - learning_rate: 0.0010
Epoch 5/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 133s 518ms/step - accuracy: 0.0121 - loss: 4.7837 - val_accuracy: 0.0113 - val_loss: 4.7847 - learning_rate: 0.0010
Epoch 6/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 135s 528ms/step - accuracy: 0.0117 - loss: 4.7822 - val_accuracy: 0.0113 - val_loss: 4.7846 - learning_rate: 0.0010
Epoch 7/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 134s 523ms/step - accura

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# ต้องใช้ preprocess_input แทน rescale=1./255
datagen2 = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← สำคัญมาก!
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data2 = datagen2.flow_from_dataframe(
    df, x_col='filename', y_col='breed',
    target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='training'
)

val_data2 = datagen2.flow_from_dataframe(
    df, x_col='filename', y_col='breed',
    target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='validation'
)

# สร้าง model ใหม่
base2 = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base2.trainable = False

model3 = models.Sequential([
    base2,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history3 = model3.fit(
    train_data2,
    validation_data=val_data2,
    epochs=20,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

Found 8178 validated image filenames belonging to 120 classes.
Found 2044 validated image filenames belonging to 120 classes.
Epoch 1/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 185s 634ms/step - accuracy: 0.5142 - loss: 2.0346 - val_accuracy: 0.7206 - val_loss: 0.9179
Epoch 2/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 547ms/step - accuracy: 0.6920 - loss: 1.1550 - val_accuracy: 0.7471 - val_loss: 0.9009
Epoch 3/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 137s 536ms/step - accuracy: 0.7375 - loss: 0.9706 - val_accuracy: 0.7529 - val_loss: 0.8986
Epoch 4/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 137s 536ms/step - accuracy: 0.7651 - loss: 0.8223 - val_accuracy: 0.7745 - val_loss: 0.8332
Epoch 5/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 546ms/step - accuracy: 0.7845 - loss: 0.7360 - val_accuracy: 0.7686 - val_loss: 0.8359
Epoch 6/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 137s 535ms/step - accuracy: 0.7996 - loss: 0.6602 - val_accuracy: 0.7593 - val_loss: 0.8613
Epoch 7/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 137s 534ms/step - accuracy: 0.8156 - loss:

In [ ]:
# Unfreeze layer ท้ายๆ ของ EfficientNetB0
for layer in base2.layers[-50:]:
    layer.trainable = True

model3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history4 = model3.fit(
    train_data2,
    validation_data=val_data2,
    epochs=15,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

Epoch 1/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 204s 668ms/step - accuracy: 0.5784 - loss: 1.5780 - val_accuracy: 0.6830 - val_loss: 1.1506
Epoch 2/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 142s 556ms/step - accuracy: 0.6203 - loss: 1.3588 - val_accuracy: 0.6952 - val_loss: 1.0738
Epoch 3/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 141s 551ms/step - accuracy: 0.6705 - loss: 1.1779 - val_accuracy: 0.7118 - val_loss: 1.0232
Epoch 4/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 138s 539ms/step - accuracy: 0.6892 - loss: 1.0881 - val_accuracy: 0.7343 - val_loss: 0.9509
Epoch 5/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 140s 546ms/step - accuracy: 0.7158 - loss: 0.9619 - val_accuracy: 0.7422 - val_loss: 0.8880
Epoch 6/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 141s 550ms/step - accuracy: 0.7261 - loss: 0.9334 - val_accuracy: 0.7495 - val_loss: 0.8832
Epoch 7/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 143s 560ms/step - accuracy: 0.7392 - loss: 0.8587 - val_accuracy: 0.7637 - val_loss: 0.8233
Epoch 8/15
256/256 ━━━━━━━━━━━━━━━━━━━━ 136s 532ms/step - accuracy: 0.7530 -

In [ ]:
!pip install tensorflowjs

import tensorflowjs as tfjs
import json

# แปลง model
tfjs.converters.save_keras_model(model3, "./tfjs_model")

# บันทึก class names
class_names = list(train_data2.class_indices.keys())
with open("./tfjs_model/class_names.json", "w") as f:
    json.dump(class_names, f)

print(f"✅ เสร็จแล้ว! {len(class_names)} สายพันธุ์")

INFO: pip is looking at multiple versions of wheel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.40.1 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
xarray 2025.12.0 requires pac

failed to lookup keras version from the file,
    this is likely a weight only file
weight normalization_2/count with shape () and dtype int64 was auto converted to the type int32
✅ เสร็จแล้ว! 120 สายพันธุ์


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("tfjs_model", "zip", "./tfjs_model")
files.download("tfjs_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

เทรนต่อ Val accuracy อยู่ที่ 78%--> 90%


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub, os, json, zipfile, pandas as pd

# 1. โหลด dataset
path = kagglehub.dataset_download("dilakshanchandrasena/dog-breed-classification")
DATA_DIR = path + "/DATASET"
TRAIN_DIR = DATA_DIR + "/train_images"
df = pd.read_csv(DATA_DIR + "/train_data.csv")
df['filename'] = TRAIN_DIR + "/" + df['id'] + ".jpg"

# 2. เตรียมข้อมูล
datagen2 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)
train_data2 = datagen2.flow_from_dataframe(
    df, x_col='filename', y_col='breed',
    target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='training'
)
val_data2 = datagen2.flow_from_dataframe(
    df, x_col='filename', y_col='breed',
    target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='validation'
)
NUM_CLASSES = len(train_data2.class_indices)

# 3. โหลด model จาก Drive
from google.colab import drive
drive.mount('/content/drive')

with zipfile.ZipFile("/content/drive/MyDrive/tfjs_model.zip", 'r') as z:
    z.extractall("/content/tfjs_model")

# โหลด model กลับมาเป็น Keras
# ต้องใช้ไฟล์ .h5 แทน — มีเซฟไว้ไหม?

100%|██████████| 689M/689M [00:06<00:00, 117MB/s]

Extracting files...


Found 8178 validated image filenames belonging to 120 classes.
Found 2044 validated image filenames belonging to 120 classes.
Mounted at /content/drive


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# สร้าง model
base2 = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base2.trainable = False

model3 = models.Sequential([
    base2,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train พร้อมเซฟอัตโนมัติลง Drive
history3 = model3.fit(
    train_data2,
    validation_data=val_data2,
    epochs=20,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True),
        ModelCheckpoint(
            "/content/drive/MyDrive/dog_breed_best.keras",
            save_best_only=True,
            monitor='val_accuracy'
        )
    ]
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 194s 644ms/step - accuracy: 0.5046 - loss: 2.0659 - val_accuracy: 0.7314 - val_loss: 0.9104
Epoch 2/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 130s 508ms/step - accuracy: 0.6948 - loss: 1.1292 - val_accuracy: 0.7485 - val_loss: 0.8964
Epoch 3/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 128s 498ms/step - accuracy: 0.7307 - loss: 0.9574 - val_accuracy: 0.7461 - val_loss: 0.8596
Epoch 4/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 127s 497ms/step - accuracy: 0.7525 - loss: 0.8801 - val_accuracy: 0.7500 - val_loss: 0.8974
Epoch 5/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 128s 500ms/step - accuracy: 0.7839 - loss: 0.7349 - val_accuracy: 0.7691 - val_loss: 0.8275
Epoch 6/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 130s 507ms/step - accuracy: 0.8069 - loss: 0.6345 - val_accuracy: 0.7608 - val_loss: 0.8624
Epoch 7/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 143s 511ms/step - accuracy: 0.8151 - loss: 0.5902 - val_accuracy: 0.7740 - val_loss: 0.8278
Epoch 8/20
256/256 ━━━━━━

In [ ]:
import os
for f in os.listdir("/content/drive/MyDrive"):
    if 'dog_breed' in f:
        print(f)

dog_breed_best.keras


In [ ]:
!pip install tensorflowjs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.40.1 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [ ]:
import tensorflowjs as tfjs
import json, shutil, zipfile
from tensorflow.keras.models import load_model

# โหลด model จาก Drive
model_best = load_model("/content/drive/MyDrive/dog_breed_best.keras")
print("✅ โหลด model แล้ว!")

# แปลงเป็น TensorFlow.js
tfjs.converters.save_keras_model(model_best, "./tfjs_model_final")

# เซฟ class names
with open("./tfjs_model_final/class_names.json", "w") as f:
    json.dump(list(train_data2.class_indices.keys()), f)

# zip และเซฟลง Drive
shutil.make_archive("tfjs_model_final", "zip", "./tfjs_model_final")
shutil.copy("tfjs_model_final.zip", "/content/drive/MyDrive/tfjs_model_final.zip")
print("✅ เซฟลง Drive แล้ว!")

✅ โหลด model แล้ว!
failed to lookup keras version from the file,
    this is likely a weight only file
weight normalization/count with shape () and dtype int64 was auto converted to the type int32
✅ เซฟลง Drive แล้ว!


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/tfjs_model_final.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 1: ติดตั้ง
!pip install tensorflowjs

# Cell 2: convert
import tensorflowjs as tfjs
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/dog_breed_best.keras")
tfjs.converters.save_keras_model(model, "./tfjs_graph_model")

# Cell 3: zip และ download
import shutil
from google.colab import files

shutil.make_archive("tfjs_graph_model", "zip", "./tfjs_graph_model")
files.download("tfjs_graph_model.zip")

INFO: pip is looking at multiple versions of wheel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
google-cloud-bigquery 3.40.1 requires packa

KeyboardInterrupt: 

In [ ]:
# Cell 1: ติดตั้งก่อน
!pip install tensorflowjs


In [ ]:
# Cell 2: restart แล้วค่อยรัน cell นี้
import tensorflowjs as tfjs
from tf_keras.models import load_model  # ใช้ tf_keras แทน tensorflow.keras

model = load_model("/content/drive/MyDrive/dog_breed_best.keras")
tfjs.converters.save_keras_model(model, "./tfjs_graph_model")

OSError: No file or directory found at /content/drive/MyDrive/dog_breed_best.keras

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# หาไฟล์ .keras ทั้งหมด
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.keras'):
            print(os.path.join(root, file))

Mounted at /content/drive
/content/drive/MyDrive/dog_breed_best.keras


In [ ]:
import tensorflowjs as tfjs
import keras

model = keras.models.load_model("/content/drive/MyDrive/dog_breed_best.keras")
tfjs.converters.save_keras_model(model, "./tfjs_graph_model")

failed to lookup keras version from the file,
    this is likely a weight only file
weight normalization/count with shape () and dtype int64 was auto converted to the type int32


In [ ]:
# zip และ download
import shutil
from google.colab import files

shutil.make_archive("tfjs_graph_model", "zip", "./tfjs_graph_model")
files.download("tfjs_graph_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflowjs as tfjs
import keras
import tensorflow as tf

# โหลดโมเดล
model = keras.models.load_model("/content/drive/MyDrive/dog_breed_best.keras")

# Save เป็น SavedModel format ก่อน
model.export("./saved_model")

# แล้วค่อย convert เป็น tfjs
!tensorflowjs_converter \
  --input_format=tf_saved_model \
  --output_format=tfjs_graph_model \
  ./saved_model \
  ./tfjs_graph_model_v2

Saved artifact at './saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 120), dtype=tf.float32, name=None)
Captures:
  139965247785488: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139965247789136: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139965748466448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748465680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748461840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748467216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748467792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748467984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748462608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139965748467408: TensorSpec(shape=(), dtype=tf.resource, name=None

In [ ]:
# zip และ download
import shutil
from google.colab import files

shutil.make_archive("tfjs_graph_model_v2", "zip", "./tfjs_graph_model_v2")
files.download("tfjs_graph_model_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task
สร้างโมเดลจำแนกสายพันธุ์สุนัขจากภาพโดยใช้เทคนิค Transfer Learning และ Convolutional Neural Networks (CNNs) จากชุดข้อมูล Kaggle "dilakshanchandrasena/dog-breed-classification" เปรียบเทียบประสิทธิภาพของ MobileNetV2 และ EfficientNetB0, แก้ไขปัญหา Initial low accuracy ที่พบในการใช้ EfficientNetB0 ด้วยการปรับปรุง Preprocessing, และแปลงโมเดลที่ได้ให้เป็นรูปแบบ TensorFlow.js พร้อมทั้งจัดทำโครงสร้างเนื้อหาสไลด์สำหรับนำเสนอโครงการนี้

## สรุปภาพรวมโครงการ

### Subtask:
อธิบายวัตถุประสงค์หลักของโปรเจกต์นี้ ซึ่งคือการจำแนกสายพันธุ์สุนัขจากภาพ โดยใช้เทคนิค Transfer Learning และ Convolutional Neural Networks (CNNs) ระบุแหล่งที่มาของข้อมูล (Kaggle dataset) และเป้าหมายของโมเดลที่พัฒนาขึ้น


## สรุปภาพรวมโครงการ

### วัตถุประสงค์หลักของโครงการ
โครงการนี้มีวัตถุประสงค์หลักเพื่อพัฒนาโมเดลสำหรับการจำแนกสายพันธุ์สุนัขจากรูปภาพ โดยใช้เทคนิคด้านปัญญาประดิษฐ์และ Machine Learning

### เทคนิคที่ใช้
เราจะใช้เทคนิค **Transfer Learning** ซึ่งเป็นการนำโมเดลโครงข่ายประสาทเทียม (Convolutional Neural Networks หรือ CNNs) ที่ได้รับการฝึกฝนมาแล้วด้วยชุดข้อมูลขนาดใหญ่สำหรับงานจำแนกรูปภาพทั่วไป (เช่น ImageNet) มาปรับใช้กับงานเฉพาะทางของเรา ทำให้สามารถลดระยะเวลาและทรัพยากรในการฝึกฝนโมเดลลงได้มาก และยังช่วยเพิ่มประสิทธิภาพของโมเดลอีกด้วย

### แหล่งที่มาของข้อมูล
ชุดข้อมูลที่ใช้ในการฝึกฝนและทดสอบโมเดลมาจาก **Kaggle dataset** ชื่อ `dilakshanchandrasena/dog-breed-classification` ซึ่งประกอบด้วยรูปภาพสุนัขหลากหลายสายพันธุ์พร้อมป้ายกำกับที่ถูกต้อง

### เป้าหมายของโมเดล
เป้าหมายสูงสุดคือการสร้างโมเดลที่มีความแม่นยำสูงในการระบุสายพันธุ์สุนัขจากรูปภาพที่ป้อนเข้ามา เพื่อให้สามารถนำไปประยุกต์ใช้ในงานจริง เช่น แอปพลิเคชันระบุสายพันธุ์สัตว์เลี้ยง หรือระบบจัดการข้อมูลสัตว์

## รวบรวมขั้นตอนและเครื่องมือ

### Subtask:
แจกแจงขั้นตอนหลักของโปรเจกต์ทีละขั้นตอน โดยระบุเครื่องมือ (ไลบรารี Python) ที่ใช้ในแต่ละช่วง


## รวบรวมขั้นตอนและเครื่องมือ

### Subtask:
แจกแจงขั้นตอนหลักของโปรเจกต์ทีละขั้นตอน โดยระบุเครื่องมือ (ไลบรารี Python) ที่ใช้ในแต่ละช่วง

*   **1. การดาวน์โหลดและเตรียมข้อมูล**
    *   `kagglehub`: ใช้สำหรับดาวน์โหลดชุดข้อมูลจาก Kaggle.
    *   `os`: ใช้สำหรับการจัดการไฟล์และไดเรกทอรี.
    *   `pandas`: ใช้สำหรับการโหลดและจัดการข้อมูล `train_data.csv`.
    *   `ImageDataGenerator` (จาก `tensorflow.keras.preprocessing.image`): ใช้สำหรับการทำ Data Augmentation และการสร้าง Data Pipeline สำหรับการฝึกและตรวจสอบข้อมูล.

*   **2. การสร้างและเทรนโมเดลแรก (MobileNetV2)**
    *   `MobileNetV2` (จาก `tensorflow.keras.applications`): ใช้เป็น Base Model สำหรับ Transfer Learning.
    *   `layers` และ `models` (จาก `tensorflow.keras`): ใช้สำหรับสร้างโครงสร้างโมเดลเพิ่มเติม (เช่น GlobalAveragePooling2D, Dense layers, Dropout).
    *   `EarlyStopping` (จาก `tensorflow.keras.callbacks`): ใช้สำหรับหยุดการเทรนก่อนกำหนดเมื่อประสิทธิภาพของโมเดลบนชุดข้อมูล validation ไม่ดีขึ้น.

*   **3. การ Fine-tune โมเดลแรก**
    *   `base.trainable = True` และ `for layer in base.layers[:-30]: layer.trainable = False`: ใช้เพื่อ Unfreeze (ปลดล็อค) เลเยอร์ท้ายๆ ของ Base Model เพื่อ Fine-tune.
    *   `tf.keras.optimizers.Adam(learning_rate=1e-5)`: ใช้ Optimizer แบบ Adam พร้อม Learning Rate ที่ต่ำลงสำหรับการ Fine-tuning.

*   **4. การสร้างและเทรนโมเดลที่สอง (EfficientNetB0) และการแก้ไขปัญหา Initial low accuracy**
    *   `EfficientNetB0` (จาก `tensorflow.keras.applications`): ใช้เป็น Base Model ใหม่สำหรับ Transfer Learning.
    *   `layers` และ `models` (จาก `tensorflow.keras`): ใช้สำหรับสร้างโครงสร้างโมเดลเพิ่มเติม (เช่น GlobalAveragePooling2D, BatchNormalization, Dense layers, Dropout).
    *   `EarlyStopping` (จาก `tensorflow.keras.callbacks`): ใช้สำหรับหยุดการเทรนก่อนกำหนด.
    *   `ReduceLROnPlateau` (จาก `tensorflow.keras.callbacks`): ใช้สำหรับปรับ Learning Rate โดยอัตโนมัติเมื่อประสิทธิภาพของโมเดลไม่ดีขึ้น.
    *   `preprocess_input` (จาก `tensorflow.keras.applications.efficientnet`): **สำคัญมาก!** ใช้ร่วมกับ `ImageDataGenerator` เพื่อเตรียมข้อมูลภาพให้ถูกต้องตามที่ EfficientNetB0 ต้องการ ซึ่งช่วยแก้ปัญหา `Initial low accuracy` ที่พบใน EfficientNetB0 เมื่อไม่ได้ใช้ Preprocessing ที่ถูกต้อง.

*   **5. การ Fine-tune โมเดลที่สอง**
    *   `base2.trainable = True` และ `for layer in base2.layers[-50:]: layer.trainable = True`: ใช้เพื่อ Unfreeze (ปลดล็อค) เลเยอร์ท้ายๆ ของ Base Model EfficientNetB0 เพื่อ Fine-tune.
    *   `tf.keras.optimizers.Adam(learning_rate=1e-5)`: ใช้ Optimizer แบบ Adam พร้อม Learning Rate ที่ต่ำลงสำหรับการ Fine-tuning.
    *   `ModelCheckpoint` (จาก `tensorflow.keras.callbacks`): ใช้สำหรับบันทึกโมเดลที่มีประสิทธิภาพดีที่สุดบนชุดข้อมูล validation ลงใน Google Drive โดยอัตโนมัติ.

*   **6. การแปลงโมเดลเป็น TensorFlow.js และการบันทึก**
    *   `tensorflowjs`: ไลบรารีหลักสำหรับการแปลงโมเดล Keras ไปยังรูปแบบ TensorFlow.js.
    *   `json`: ใช้สำหรับบันทึก Class Names ในรูปแบบ JSON.
    *   `shutil`: ใช้สำหรับการบีบอัดไฟล์ (zip) โฟลเดอร์ที่แปลงแล้ว.
    *   `google.colab.files`: ใช้สำหรับการดาวน์โหลดไฟล์ที่บีบอัดแล้วไปยังเครื่องคอมพิวเตอร์ของผู้ใช้.
    *   `keras.models.load_model` (หรือ `tf_keras.models.load_model`): ใช้สำหรับโหลดโมเดล Keras ที่บันทึกไว้ (`.keras` format) เพื่อเตรียมแปลง.
    *   `model.export("./saved_model")` (จาก `keras`): ใช้สำหรับ Export โมเดล Keras ไปยังรูปแบบ TensorFlow SavedModel.
    *   `!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model ./saved_model ./tfjs_graph_model_v2`: คำสั่ง Command Line ในการแปลงโมเดลที่ Export เป็น SavedModel format ไปยังรูปแบบ TensorFlow.js Graph Model.

## วิเคราะห์ผลลัพธ์และแนวทางแก้ไขปัญหา

### Subtask:
วิเคราะห์ผลลัพธ์ด้านความแม่นยำ (accuracy) จากการทดลองแต่ละครั้ง, ชี้ให้เห็นถึงปัญหา 'Initial low accuracy' ของ EfficientNetB0 และอธิบายวิธีแก้ไขโดยการใช้ `preprocess_input` ที่ถูกต้อง
